In [1]:
from surprise import Reader, Dataset, KNNWithMeans, accuracy
from surprise.model_selection import train_test_split


# Load movielens1M dataset
reader = Reader(line_format='user item rating timestamp', sep='::', rating_scale=(1,5))
data = Dataset.load_from_file('../artifacts/ratings.dat', reader=reader)

# Partition
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Train model
opts = {'name': 'pearson', 'user_based': True}
algo = KNNWithMeans(k=20, min_k=3, sim_options=opts)
algo.fit(trainset)

# Predict
predictions = algo.test(testset)

# Evaluate
accuracy.rmse(predictions)

Computing the pearson similarity matrix...
Done computing similarity matrix.
RMSE: 0.9333


0.9332519850120894

In [2]:
# Make predictions on user 196
uid = str(196)
iid = str(302)
pred = algo.predict(uid, iid, r_ui=4.0, verbose=True)
print(pred)

user: 196        item: 302        r_ui = 4.00   est = 4.49   {'actual_k': 20, 'was_impossible': False}
user: 196        item: 302        r_ui = 4.00   est = 4.49   {'actual_k': 20, 'was_impossible': False}


In [3]:
# Tune algorithm parameters with GridSearchCV
from surprise.model_selection import GridSearchCV


param_grid = {'k': [10, 20, 40, 50], 'min_k': [1, 3, 5, 10], 'sim_options': {'name': ['pearson'], 'user_based': [True]}}
gs = GridSearchCV(KNNWithMeans, param_grid, measures=['rmse'], cv=3)
gs.fit(data)

# Best score
print(gs.best_score['rmse'])

# Best combination of parameters
print(gs.best_params['rmse'])

Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.

In [7]:
from pathlib import Path
from joblib import dump, load

# Train model with the best parameters and save it
algo = KNNWithMeans(k=gs.best_params['rmse']['k'], min_k=gs.best_params['rmse']['min_k'], sim_options=gs.best_params['rmse']['sim_options'])
algo.fit(trainset)
dump(algo, Path("../artifacts/cf_model_uu_pearson.joblib"), compress=3)

# Load best model and test it to make predictions on user 196
best_model = load(Path("../artifacts/cf_model_uu_pearson.joblib"))
uid = str(196)
iid = str(302)
pred = best_model.predict(uid, iid, r_ui=4.0, verbose=True)
print(pred)

Computing the pearson similarity matrix...
Done computing similarity matrix.
user: 196        item: 302        r_ui = 4.00   est = 4.51   {'actual_k': 32, 'was_impossible': False}
user: 196        item: 302        r_ui = 4.00   est = 4.51   {'actual_k': 32, 'was_impossible': False}
